<a href="https://colab.research.google.com/github/muneer-ahmad10/Natural-Language-processing/blob/main/Mini_Exercise_Cosine_Similarity_Faiss_Chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 50.2 MB/s eta 0:00:00


In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
sentences = [
    "Machine Learning Engineer",
    "AI Engineer",
    "Football Player",
    "Data Scientist"
]

In [4]:
embeddings = model.encode(sentences)

In [5]:
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(
            [embeddings[i]],
            [embeddings[j]]
        )[0][0]

        print(
            f"{sentences[i]} <-> {sentences[j]} = {sim:.2f}"
        )

Machine Learning Engineer <-> AI Engineer = 0.73
Machine Learning Engineer <-> Football Player = 0.23
Machine Learning Engineer <-> Data Scientist = 0.61
AI Engineer <-> Football Player = 0.25
AI Engineer <-> Data Scientist = 0.46
Football Player <-> Data Scientist = 0.25


**Why can embeddings understand similarity?**

Embeddings represent words or sentences as dense numerical vectors. During training, semantically related concepts appear in similar contexts, causing their vectors to occupy nearby regions in vector space. Similarity metrics such as cosine similarity can then measure how close their meanings are.

# **First Chunking Code**

In [6]:
text = """
Attention is a mechanism that allows a model
to focus on important words.

Transformers use self-attention to process
all words simultaneously.

BERT is an encoder-only transformer.

GPT is a decoder-only transformer.
"""

In [7]:
chunks=text.split("\n")
print(chunks)

['', 'Attention is a mechanism that allows a model', 'to focus on important words.', '', 'Transformers use self-attention to process', 'all words simultaneously.', '', 'BERT is an encoder-only transformer.', '', 'GPT is a decoder-only transformer.', '']


In [8]:
chunk_embedding=model.encode(chunks)

In [10]:
chunk_embedding.shape

(11, 384)

In [13]:
import faiss
import numpy as np

dimension = chunk_embedding.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(chunk_embedding).astype("float32")
)

In [14]:
question = "What is self attention?"

In [15]:
question_embedding = model.encode([question])

In [16]:
distances, indices = index.search(
    np.array(question_embedding).astype("float32"),
    k=1
)

In [19]:
print(chunks[indices[0][0]])

Attention is a mechanism that allows a model


In [20]:
question = "Which transformer uses only the decoder?"

In [21]:
question_embedding = model.encode([question])

In [22]:
distances, indices = index.search(
    np.array(question_embedding).astype("float32"),
    k=1
)

In [23]:
print(chunks[indices[0][0]])

GPT is a decoder-only transformer.
